In [1]:
def set_up_optimization(event,budget,directory=''):
    
    '''Code to set up optimization parameters. Called by optimization pipeline code and used 
    in subsequent AMPL codes. This function reads the data files and sets up parameters

    budget: budgetary limit
    event: Define tornado event of interest, e.g., 'EF5' for EF5, 'Joplin' for historic Joplin tornado,
    'Weighted' for case weighted by frequency of various EF intensities
    directory: Define working directory from where to read data
    
    '''
    
    # Define global variables 
    
    global x_b_0, c_b_ret, casualty_costs, C_dr, indices_noret
    global building_size,c_b_rep, t
    global DS_matrix,DS_matrix_for_casualties
    global c_d,casualty_rates,V_db,indices,bld_indices
    global neighborhoods, permissible, f_pw, f_footprint

    # Define parameters of interest

    data_file = rf"{directory}community_data.xlsx" # Load main data file
 
    econ_params = pd.read_excel(data_file,'Econ_params') # load econ parameters for analysis

    discount_rate = econ_params[econ_params['Parameter']=='r']['Value'].item() # Discount rate

    casualty_costs = np.array(econ_params['Value'].loc[6:10]) # casualty costs for varying damage levels
        
    t = econ_params[econ_params['Parameter']=='t']['Value'].item() # number of years of study period

    f_pw = 0
    for time in np.arange(t):
        f_pw += 1/t*(1/(1+discount_rate)**time)


    # Adjust PWF to also incorporate differences in tornado frequency due to changes in time 
    # (base events assume 100 year period of interest)
    
    if event == 'Weighted':
        
        f_pw = f_pw/100*t
    
    # ====  Load neighborhoods
    
    neighborhoods = gp.read_file(rf"{directory}Neighborhoods\\neighborhoods.shp") # load neighborhoods
    
    n_neighborhoods = neighborhoods.shape[0] # number of neighborhoods
    
    # ========================================================================
    
    # ====================Buildings ==========================================
    
    # ========================================================================
    
    # ========= Load indices and sets 
    
    indices = pd.read_excel(data_file,"Indices") # indices of different buildings
    
    indices_noret = indices[indices['Retro_class']==0] # extract non-retrofitted indices
        
    # ======== Load building portfolio data
    
    json_file = open(rf"{directory}x_b_0.json", 'r')
    x_b_0 = np.array(json.load(json_file))
    json_file.close()
    
    # Extract values of residential/business/industrial in each neighborhood 
    
    total_residential = x_b_0[:,indices_noret['Class']=='Res'].sum(axis=2).sum(axis=1) # total residential buildings in neighborhood
    
    total_business = x_b_0[:,indices_noret['Class']=='Bus'].sum(axis=2).sum(axis=1) # total business buildings in neighborhood
    
    total_ind = x_b_0[:,indices_noret['Class']=='Ind'].sum(axis=2).sum(axis=1) # total industrial buildings in neighborhood

    
    # =====================================================================
    # ======================= Damage State Data ===========================
    # =====================================================================
    
    # ========= Damage state probability matrix
    
    json_file = open(rf"{directory}DS_prob_matrix{event}.json", 'r')
    DS_matrix = np.array(json.load(json_file))
    json_file.close()
    
    # ======= Damage state probability matrix (for calculating casualties, these are zeroed out for sheltered homes)
    
    json_file = open(rf"{directory}DS_prob_matrix_for_casualties{event}.json",'r')
    DS_matrix_for_casualties = np.array(json.load(json_file))
    json_file.close()

    # ====== building footprint factors (for use in weighted case)

    if event == 'Weighted':
        f_footprint = np.array(pd.read_excel(data_file,"Footprint_factors")['Factor'])
        
    else:
        f_footprint = np.ones(x_b_0.shape[1])
        
    # ==================================================================
    # ========================= Dislocations ===========================
    # ==================================================================

    json_file = open(rf"{directory}restoration_costs{event}.json",'r')
    c_d = np.array(json.load(json_file))
    json_file.close()
    
    # ==================================================================
    # ========================= Casualties = ===========================
    # ==================================================================
    
    json_file = open(rf"{directory}casualty_rates.json",'r')
    casualty_rates = np.array(json.load(json_file))
    json_file.close()
    
    # adding in casualty rate for DS0...
    casualty_rates = np.around(np.vstack(([0,0,0,0],casualty_rates)).astype(float),5) # casualty rate for DS and injurity severity level
    
    # Get building size in terms of residents inside during tornado
    
    json_file = open(rf"{directory}residents_per_bld.json",'r')
    building_size = np.array(json.load(json_file)).T # number of residents per building type in each neighborhood during tornado
    json_file.close()
    
    # =================================================================
    # ================ Repair and retrofit costs ======================
    # =================================================================
    
    DS_fractions = np.array([0,0.02,0.1,0.5,1]) # fraction of home value lost in given damage state

    building_costs = np.array(pd.read_excel(data_file,'Building_Costs')['Cost']) # load home base value

    c_b_rep =  building_costs[:,np.newaxis]*DS_fractions # repair cost matrix based on damage state

    retrofit_data = pd.read_excel(data_file,"Permissible Upgrade") # load permissible retrofits

    retro_class = indices['Retro_class'] # get retrofit class 0-5
    archetype = indices['Code'] # get archetype class 0-5

    # initialize building retrofit cost

    c_b_ret = np.zeros([archetype.max()+1,retro_class.max()+1,retro_class.max()+1]) 
    
    # Get retrofit costs
    # Cost as proportion of building value from transitioning from a to b (e.g., costs 0.1 to go from 0 to 1)

    c_r0_r1 = econ_params['Value'][3].item()
    c_r0_r2 = econ_params['Value'][4].item()
    c_r1_r2 = econ_params['Value'][5].item()
    
    ret_cost_matrix = np.array([[0,c_r0_r1,c_r0_r2,0,c_r0_r1,c_r0_r2],
                           [0,0,c_r1_r2,0,0,c_r1_r2],
                           [0,0,0,0,0,0],
                           [0,0,0,0,c_r0_r1,c_r0_r2],
                           [0,0,0,0,0,c_r1_r2],
                           [0,0,0,0,0,0]])
    
    shelter_cost = econ_params['Value'][2].item()
    
    # Binary whether the retrofit includes shelter installation (e.g., going from 0 to 3 involves shelter)
    
    shelter_binary_matrix = np.array([[0,0,0,1,1,1],
                                     [0,0,0,1,1,1],
                                     [0,0,0,1,1,1],
                                     [0,0,0,0,0,0],
                                     [0,0,0,0,0,0],
                                     [0,0,0,0,0,0]])

    # For each permissible retrofit, populate the matrix
    
    for i in np.arange(retrofit_data.shape[0]):
        r_to = retrofit_data.iloc[i]['To']
        r_from = retrofit_data.iloc[i]['From']
        code = retrofit_data.iloc[i]['Code']
        c_b_ret[code,r_from,r_to] = ret_cost_matrix[r_from,r_to]*c_b_rep[code,4]+shelter_cost*shelter_binary_matrix[r_from,r_to]

    # ===================== Permissible upgrades ==========================
    
    permissible = retrofit_data[['Code','From','To']]+1
    permissible = list(permissible.itertuples(index=False, name=None))
    
    # ==================================================================
    # ========================== Debris ================================
    # ==================================================================
    
    C_dr = 33 # assumed cost of debris removal per cubic meter
    
    # load debris predictions
    
    json_file = open(rf"{directory}building_debris{event}.json", 'r')
    V_db = np.array(json.load(json_file)) # debris volume
    json_file.close()
    